# 9 WorkFlow Analista Jr

### 9.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán
<br>El Analista Jr corre sus scripts en la virtual manchine **desktop-jr** que tiene estas características


*   Normal, paga tarifa completa, nunca es apagada por Google
*   reside en el datacenter de Toronto, Canada
*   64 GB de memoria RAM
*   8 vCPU


En Analista Jr **no** puede utilizar Google Colab porque los 12 GB de dichas maquinas virtuales no son suficientes para el tamaño del dataset que está utilizando.



## 9.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Tue Sep 15 17:09:59 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671183,35.9,1479497,79.1,1479497,79.1
Vcells,1242599,9.5,8388608,64.0,1978697,15.1


In [3]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, u

#### Parametros

In [4]:
PARAM <- list()
PARAM$semilla_primigenia <- 140009

PARAM$experimento <- 9200
PARAM$dataset <- "analistajr_competencia_2026.csv.gz"

#### Carpeta del Experimento

In [5]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 9.3.1   Preprocesamiento del dataset

#### 9.3.1.1  DT incorporar dataset

In [6]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 9.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [7]:
if( !require("mice")) install.packages("mice", repos = "http://cran.us.r-project.org")
require("mice")

Loading required package: mice


Attaching package: ‘mice’


The following object is masked from ‘package:stats’:

    filter


The following objects are masked from ‘package:base’:

    cbind, rbind




In [8]:
# Escrito por alumnos de  Universidad Austral  Rosario

Corregir_MICE <- function(pcampo, pmeses) {

  meth <- rep("", ncol(dataset))
  names(meth) <- colnames(dataset)
  meth[names(meth) == pcampo] <- "sample"

  # llamada a mice  !
  imputacion <- mice(dataset,
    method = meth,
    maxit = 5,
    m = 1,
    seed = 7)

  tbl <- mice::complete(dataset)

  dataset[, paste0(pcampo) := ifelse(foto_mes %in% pmeses, tbl[, get(pcampo)], get(pcampo))]

}


In [9]:
Corregir_interpolar <- function(pcampo, pmeses) {

  tbl <- dataset[, list(
    "v1" = shift(get(pcampo), 1, type = "lag"),
    "v2" = shift(get(pcampo), 1, type = "lead")
  ),
  by = eval(envg$PARAM$dataset_metadata$entity_id)
  ]

  tbl[, paste0(envg$PARAM$dataset_metadata$entity_id) := NULL]
  tbl[, promedio := rowMeans(tbl, na.rm = TRUE)]

  dataset[
    ,
    paste0(pcampo) := ifelse(!(foto_mes %in% pmeses),
      get(pcampo),
      tbl$promedio
    )
  ]
}

In [10]:
AsignarNA_campomeses <- function(pcampo, pmeses) {

  if( pcampo %in% colnames( dataset ) ) {

    dataset[ foto_mes %in% pmeses, paste0(pcampo) := NA ]
  }
}

In [11]:

Corregir_atributo <- function(pcampo, pmeses, pmetodo)
{
  # si el campo no existe en el dataset, Afuera !
  if( !(pcampo %in% colnames( dataset )) )
    return( 1 )

  # llamo a la funcion especializada que corresponde
  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
    "EstadisticaClasica"  = Corregir_interpolar(pcampo, pmeses),
    "MICE"                = Corregir_MICE(pcampo, pmeses),
  )

  return( 0 )
}

In [12]:

Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  cat( "inicio Corregir_Rotas()\n")
  # acomodo los errores del dataset

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  cat( "fin Corregir_rotas()\n")
}


In [13]:
# resuelvo el Catastrophe Analysis

setorder( dataset, numero_de_cliente, foto_mes )

PARAM$CA$metodo= "MachineLearning"

if( PARAM$CA$metodo %in% c("MachineLearning", "EstadisticaClasica", "MICE") )
  Corregir_Rotas(dataset, PARAM$CA$metodo)

inicio Corregir_Rotas()
fin Corregir_rotas()


#### 9.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, ajustando por algunos indices financieros

In [14]:
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)


In [15]:
# los valores que siguen fueron calculados por alumnos

# momento 1.0  31-dic-2020 a las 23:59
vIPC <- c(
  1.9903030878, 1.9174403544, 1.8296186587,
  1.7728862972, 1.7212488323, 1.6776304408,
  1.6431248196, 1.5814483345, 1.4947526791,
  1.4484037589, 1.3913580777, 1.3404220402,
  1.3154288912, 1.2921698342, 1.2472681797,
  1.2300475145, 1.2118694724, 1.1881073259,
  1.1693969743, 1.1375456949, 1.1065619600,
  1.0681100000, 1.0370000000, 1.0000000000,
  0.9680542110, 0.9344152616, 0.8882274350,
  0.8532444140, 0.8251880213, 0.8003763543,
  0.7763107219, 0.7566381305, 0.7289384687
)

vdolar_blue <- c(
   39.045455,  38.402500,  41.639474,
   44.274737,  46.095455,  45.063333,
   43.983333,  54.842857,  61.059524,
   65.545455,  66.750000,  72.368421,
   77.477273,  78.191667,  82.434211,
  101.087500, 126.236842, 125.857143,
  130.782609, 133.400000, 137.954545,
  170.619048, 160.400000, 153.052632,
  157.900000, 149.380952, 143.615385,
  146.250000, 153.550000, 162.000000,
  178.478261, 180.878788, 184.357143
)

vdolar_oficial <- c(
   38.430000,  39.428000,  42.542105,
   44.354211,  46.088636,  44.955000,
   43.751429,  54.650476,  58.790000,
   61.403182,  63.012632,  63.011579,
   62.983636,  63.580556,  65.200000,
   67.872000,  70.047895,  72.520952,
   75.324286,  77.488500,  79.430909,
   83.134762,  85.484737,  88.181667,
   91.474000,  93.997778,  96.635909,
   98.526000,  99.613158, 100.619048,
  101.619048, 102.569048, 103.781818
)

vUVA <- c(
  2.001408838932958,  1.950325472789153,  1.89323032351521,
  1.8247220405493787, 1.746027787673673,  1.6871348409529485,
  1.6361678865622313, 1.5927529755859773, 1.5549162794128493,
  1.4949100586391746, 1.4197729500774545, 1.3678188186372326,
  1.3136508617223726, 1.2690535173062818, 1.2381595983200178,
  1.211656735577568,  1.1770808941405335, 1.1570338657445522,
  1.1388769475653255, 1.1156993751209352, 1.093638313080772,
  1.0657171590878205, 1.0362173587708712, 1.0,
  0.9669867858358365, 0.9323750098728378, 0.8958202912590305,
  0.8631993702994263, 0.8253893405524657, 0.7928918905364516,
  0.7666323845128089, 0.7428976357662823, 0.721615762047849
)


In [16]:
tb_indices <- as.data.table( list(
  "IPC" = vIPC,
  "dolar_blue" = vdolar_blue,
  "dolar_oficial" = vdolar_oficial,
  "UVA" = vUVA
  )
)

tb_indices[[ 'foto_mes' ]] <- vfoto_mes

tb_indices

IPC,dolar_blue,dolar_oficial,UVA,foto_mes
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1.9903031,39.04545,38.43000,2.0014088,201901
1.9174404,38.40250,39.42800,1.9503255,201902
1.8296187,41.63947,42.54210,1.8932303,201903
1.7728863,44.27474,44.35421,1.8247220,201904
1.7212488,46.09546,46.08864,1.7460278,201905
1.6776304,45.06333,44.95500,1.6871348,201906
1.6431248,43.98333,43.75143,1.6361679,201907
1.5814483,54.84286,54.65048,1.5927530,201908
1.4947527,61.05952,58.79000,1.5549163,201909


In [17]:
drift_UVA <- function(campos_monetarios) {
  cat( "inicio drift_UVA()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.UVA,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_UVA()\n")
}


In [18]:
drift_dolar_oficial <- function(campos_monetarios) {
  cat( "inicio drift_dolar_oficial()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_oficial,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_oficial()\n")
}


In [19]:
drift_dolar_blue <- function(campos_monetarios) {
  cat( "inicio drift_dolar_blue()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_blue,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_blue()\n")
}


In [20]:
drift_deflacion <- function(campos_monetarios) {
  cat( "inicio drift_deflacion()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.IPC,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_deflacion()\n")
}


In [21]:
drift_rank_simple <- function(campos_drift) {

  cat( "inicio drift_rank_simple()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_rank") :=
      (frank(get(campo), ties.method = "random") - 1) / (.N - 1), by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat( "fin drift_rank_simple()\n")
}


In [22]:
# El cero se transforma en cero
# los positivos se rankean por su lado
# los negativos se rankean por su lado

drift_rank_cero_fijo <- function(campos_drift) {

  cat( "inicio drift_rank_cero_fijo()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[get(campo) == 0, paste0(campo, "_rank") := 0]
    dataset[get(campo) > 0, paste0(campo, "_rank") :=
      frank(get(campo), ties.method = "random") / .N, by = list(foto_mes)]

    dataset[get(campo) < 0, paste0(campo, "_rank") :=
      -frank(-get(campo), ties.method = "random") / .N, by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat("\n")
  cat( "fin drift_rank_cero_fijo()\n")
}


In [23]:
drift_estandarizar <- function(campos_drift) {

  cat( "inicio drift_estandarizar()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(campo, na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  cat( "fin drift_estandarizar()\n")
}


In [24]:
# por como armé los nombres de campos,
#  estos son los campos que expresan variables monetarias
campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

campos_monetarios

[1] "mrentabilidad"                      "mrentabilidad_annual"              
 [3] "mcomisiones"                        "mactivos_margen"                   
 [5] "mpasivos_margen"                    "mcuenta_corriente"                 
 [7] "mcaja_ahorro"                       "mcuentas_saldo"                    
 [9] "mtarjeta_visa_consumo"              "mtarjeta_master_consumo"           
[11] "mprestamos_personales"              "mpayroll"                          
[13] "mttarjeta_visa_debitos_automaticos" "mcomisiones_mantenimiento"         
[15] "mtransferencias_recibidas"          "Master_mfinanciacion_limite"       
[17] "Master_msaldototal"                 "Master_mlimitecompra"              
[19] "Master_mconsumototal"               "Master_mpagominimo"                
[21] "Visa_mfinanciacion_limite"          "Visa_msaldototal"                  
[23] "Visa_mlimitecompra"                 "Visa_mconsumototal"                
[25] "Visa_mpagominimo"

In [25]:
# ejecuto el Data Drifting
setorder( dataset, numero_de_cliente, foto_mes )


PARAM$DR$metodo <- "deflacion"

switch(PARAM$DR$metodo,
  "ninguno"        = cat("No hay correccion del data drifting"),
  "rank_simple"    = drift_rank_simple(campos_monetarios),
  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios),
  "deflacion"      = drift_deflacion(campos_monetarios),
  "dolar_blue"     = drift_dolarblue(campos_monetarios),
  "dolar_oficial"  = drift_dolaroficial(campos_monetarios),
  "UVA"            = drift_UVA(campos_monetarios),
  "estandarizar"   = drift_estandarizar(campos_monetarios)
)


inicio drift_deflacion()
fin drift_deflacion()


In [26]:
colnames(dataset)

[1] "numero_de_cliente"                  "foto_mes"                          
 [3] "internet"                           "cliente_edad"                      
 [5] "cliente_antiguedad"                 "mrentabilidad"                     
 [7] "mrentabilidad_annual"               "mcomisiones"                       
 [9] "mactivos_margen"                    "mpasivos_margen"                   
[11] "cproductos"                         "mcuenta_corriente"                 
[13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
[15] "mcuentas_saldo"                     "ctarjeta_visa"                     
[17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
[19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
[21] "mtarjeta_master_consumo"            "cprestamos_personales"             
[23] "mprestamos_personales"              "cpayroll_trx"                      
[25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
[27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
[29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
[31] "ccallcenter_transacciones"          "thomebanking"                      
[33] "chomebanking_transacciones"         "ctrx_quarter"                      
[35] "Master_status"                      "Master_mfinanciacion_limite"       
[37] "Master_Fvencimiento"                "Master_msaldototal"                
[39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
[41] "Master_fechaalta"                   "Master_mconsumototal"              
[43] "Master_cconsumos"                   "Master_mpagominimo"                
[45] "Visa_status"                        "Visa_mfinanciacion_limite"         
[47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
[49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
[51] "Visa_fechaalta"                     "Visa_mconsumototal"                
[53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
[55] "clase_ternaria"

In [27]:
# se intenta corregir el data drifting utilizando algunos indices financieros

#### 9.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [28]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]


In [29]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

[1] "numero_de_cliente"                  "foto_mes"                          
 [3] "internet"                           "cliente_edad"                      
 [5] "cliente_antiguedad"                 "mrentabilidad"                     
 [7] "mrentabilidad_annual"               "mcomisiones"                       
 [9] "mactivos_margen"                    "mpasivos_margen"                   
[11] "cproductos"                         "mcuenta_corriente"                 
[13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
[15] "mcuentas_saldo"                     "ctarjeta_visa"                     
[17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
[19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
[21] "mtarjeta_master_consumo"            "cprestamos_personales"             
[23] "mprestamos_personales"              "cpayroll_trx"                      
[25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
[27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
[29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
[31] "ccallcenter_transacciones"          "thomebanking"                      
[33] "chomebanking_transacciones"         "ctrx_quarter"                      
[35] "Master_status"                      "Master_mfinanciacion_limite"       
[37] "Master_Fvencimiento"                "Master_msaldototal"                
[39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
[41] "Master_fechaalta"                   "Master_mconsumototal"              
[43] "Master_cconsumos"                   "Master_mpagominimo"                
[45] "Visa_status"                        "Visa_mfinanciacion_limite"         
[47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
[49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
[51] "Visa_fechaalta"                     "Visa_mconsumototal"                
[53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
[55] "clase_ternaria"                     "kmes"                              
[57] "mpayroll_sobre_edad"

#### 9.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

In [30]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


#### 9.3.1.4bis  FE_manual_negocio  Feature Engineering manual con criterio de negocio (v2, 12 variables)

Segunda iteracion, mas ambiciosa que la primera (6 variables, ver notebook 06 -- sin diferencia significativa). Incluye una familia nueva: contexto macroeconomico, pensada especificamente para separar deterioro individual de shock colectivo (pandemia) -- la idea que se discutio en el diagnostico de drift.

In [31]:
# ---- Familia 1: Ratios ----
dataset[, descubierto_sobre_saldo := fifelse(
  mcuentas_saldo != 0, cdescubierto_preacordado / abs(mcuentas_saldo), 0
)]

dataset[, visa_consumo_por_trx := fifelse(
  ctarjeta_visa_transacciones > 0, mtarjeta_visa_consumo / ctarjeta_visa_transacciones, 0
)]

dataset[, master_consumo_por_trx := fifelse(
  ctarjeta_master_transacciones > 0, mtarjeta_master_consumo / ctarjeta_master_transacciones, 0
)]

dataset[, visa_utilizacion_limite := fifelse(
  Visa_mlimitecompra > 0, Visa_mconsumototal / Visa_mlimitecompra, 0
)]

dataset[, master_utilizacion_limite := fifelse(
  Master_mlimitecompra > 0, Master_mconsumototal / Master_mlimitecompra, 0
)]

# ---- Familia 3: Aceleracion verdadera (delta de delta, distinto de solo delta2) ----
dataset[, ctrx_quarter_aceleracion := ctrx_quarter_delta1 - (ctrx_quarter_lag1 - ctrx_quarter_lag2)]

# ---- Familia 4: Contexto macroeconomico ----
# "el cliente cae mas o menos que el promedio de todos los clientes ese mes?"
promedios_mes <- dataset[, .(
  ctrx_quarter_promedio_mes  = mean(ctrx_quarter, na.rm = TRUE),
  consumo_visa_promedio_mes  = mean(mtarjeta_visa_consumo, na.rm = TRUE),
  saldo_promedio_mes         = mean(mcuentas_saldo, na.rm = TRUE)
), by = foto_mes]
setorder(promedios_mes, foto_mes)

# lag del promedio MENSUAL (no por cliente) -- necesario para el "cambio relativo al contexto"
promedios_mes[, ctrx_quarter_promedio_mes_lag1 := shift(ctrx_quarter_promedio_mes, 1, type = "lag")]

dataset <- merge(dataset, promedios_mes, by = "foto_mes", all.x = TRUE)
setorder(dataset, numero_de_cliente, foto_mes)  # el merge no garantiza el orden, lo restauro

dataset[, ctrx_vs_promedio_mes := fifelse(
  ctrx_quarter_promedio_mes != 0, ctrx_quarter / ctrx_quarter_promedio_mes, 0
)]

dataset[, consumo_visa_vs_promedio_mes := fifelse(
  consumo_visa_promedio_mes != 0, mtarjeta_visa_consumo / consumo_visa_promedio_mes, 0
)]

# "deterioro relativo" = deterioro individual - deterioro general del mes
# NA en 201901 (primer mes, no hay mes anterior para comparar) -- LightGBM lo maneja nativamente como missing
dataset[, ctrx_cambio_relativo_contexto :=
  ctrx_quarter_delta1 - (ctrx_quarter_promedio_mes - ctrx_quarter_promedio_mes_lag1)
]

# ---- Familia 5: Interacciones de comportamiento ----
dataset[, senial_deterioro_financiero := ctrx_quarter_delta1 * cdescubierto_preacordado_delta1]

dataset[, perdida_productos := as.integer(cproductos < cproductos_lag1)]

dataset[, perdida_productos_2m :=
  as.integer(cproductos < cproductos_lag1) + as.integer(cproductos_lag1 < cproductos_lag2)
]

variables_manuales <- c(
  "descubierto_sobre_saldo", "visa_consumo_por_trx", "master_consumo_por_trx",
  "visa_utilizacion_limite", "master_utilizacion_limite", "ctrx_quarter_aceleracion",
  "ctrx_vs_promedio_mes", "consumo_visa_vs_promedio_mes", "ctrx_cambio_relativo_contexto",
  "senial_deterioro_financiero", "perdida_productos", "perdida_productos_2m"
)
length(variables_manuales)  # debe dar 12
variables_manuales

[1] 12

[1] "descubierto_sobre_saldo"       "visa_consumo_por_trx"         
 [3] "master_consumo_por_trx"        "visa_utilizacion_limite"      
 [5] "master_utilizacion_limite"     "ctrx_quarter_aceleracion"     
 [7] "ctrx_vs_promedio_mes"          "consumo_visa_vs_promedio_mes" 
 [9] "ctrx_cambio_relativo_contexto" "senial_deterioro_financiero"  
[11] "perdida_productos"             "perdida_productos_2m"

### 9.3.2  Comparacion con GANANCIA en vez de AUC

Exploracion: en vez de comparar por AUC promedio (que pondera las ~30.000 filas del mes, cuando el negocio solo depende del orden de las primeras ~2000), se compara directamente la ganancia real a un corte FIJO de envios, calculada sobre los 3 folds internos. Mismo esquema de nested validation y Wilcoxon secuencial que los notebooks 06/07/08 -- unicamente cambia que metrica devuelve la funcion objetivo.

In [32]:
if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")
if (!require("ranger")) install.packages("ranger")
require("ranger")

PARAM$trainingstrategy$training <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)
PARAM$trainingstrategy$training_pct <- 1.0
PARAM$trainingstrategy$positivos <- c("BAJA+1", "BAJA+2")
PARAM$trainingstrategy$validacion_interna <- c(202104, 202105, 202106)
PARAM$trainingstrategy$horizonte_meses <- 2

dataset[, clase01 := ifelse(clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0)]

campos_buenos_original <- copy(setdiff(
  colnames(dataset), c("clase_ternaria", "clase01", "azar", "numero_de_cliente", variables_manuales)
))
campos_buenos_manual <- c(campos_buenos_original, variables_manuales)

length(campos_buenos_original)
length(campos_buenos_manual)

Loading required package: lightgbm

Loading required package: ranger



[1] 275

[1] 287

In [33]:
PARAM$lgbm$param_fijos <- list(
  objective          = "binary",
  metric             = "auc",
  first_metric_only  = TRUE,
  boost_from_average = TRUE,
  feature_pre_filter = FALSE,
  verbosity          = -100,
  force_row_wise     = TRUE,
  max_bin            = 31,
  learning_rate      = 0.03,
  num_iterations     = 2048,
  early_stopping_rounds = 200,
  num_leaves         = 76,
  min_data_in_leaf   = 1800,
  feature_fraction   = 0.625,
  seed               = PARAM$semilla_primigenia
)

mes_menos <- function(foto_mes, n) {
  anio <- foto_mes %/% 100
  mes  <- foto_mes %% 100
  total <- anio * 12 + (mes - 1) - n
  anio2 <- total %/% 12
  mes2  <- total %% 12 + 1
  anio2 * 100 + mes2
}

#### Matriz de ganancia -- pico de la curva de ganancia (no un corte fijo)

In [34]:
GANANCIA_ACIERTO <- 975000
GANANCIA_COSTO    <- -25000

# en vez de un corte FIJO de envios (que penaliza a la config cuyo optimo cae lejos de ese
# punto), se usa el PICO de la curva de ganancia de cada modelo -- el mejor punto de cada uno
calcular_ganancia_pico <- function(prob, clase_real) {
  ord <- order(-prob)
  clase_ordenada <- clase_real[ord]
  ganancia_unitaria <- ifelse(clase_ordenada == "BAJA+2", GANANCIA_ACIERTO, GANANCIA_COSTO)
  max(cumsum(ganancia_unitaria))
}

#### Folds internos -- version Original / Manual (guardan la matriz de validate para poder predecir despues)

In [35]:
construir_fold_ganancia <- function(meses_train, mes_validate, campos) {

  fold_train_mask    <- dataset$foto_mes %in% meses_train &
    (dataset$clase_ternaria %in% c("BAJA+1", "BAJA+2") | dataset$azar < PARAM$trainingstrategy$training_pct)
  fold_validate_mask <- dataset$foto_mes == mes_validate

  matriz_validate <- data.matrix(dataset[fold_validate_mask, campos, with = FALSE])

  list(
    dtrain = lgb.Dataset(
      data  = data.matrix(dataset[fold_train_mask, campos, with = FALSE]),
      label = dataset[fold_train_mask, clase01],
      free_raw_data = TRUE
    ),
    dvalidate = lgb.Dataset(
      data  = matriz_validate,
      label = dataset[fold_validate_mask, clase01],
      free_raw_data = TRUE
    ),
    matriz_validate     = matriz_validate,              # se mantiene aparte para poder hacer predict() despues
    clase_real_validate = dataset[fold_validate_mask, clase_ternaria]
  )
}

construir_folds_internos_ganancia <- function(semilla, campos) {
  set.seed(semilla, kind = "L'Ecuyer-CMRG")
  dataset[, azar := runif(nrow(dataset))]

  lapply(PARAM$trainingstrategy$validacion_interna, function(mes_val) {
    corte_train <- mes_menos(mes_val, PARAM$trainingstrategy$horizonte_meses)
    meses_train_fold <- PARAM$trainingstrategy$training[ PARAM$trainingstrategy$training <= corte_train ]
    construir_fold_ganancia(meses_train_fold, mes_val, campos)
  })
}

# Estimar_Ganancia_lightgbm: misma interfaz que Estimar_AUC_lightgbm (list(metrica_promedio, niter_promedio)),
# pero la metrica es GANANCIA a corte fijo en vez de AUC
Estimar_Ganancia_lightgbm <- function(x, folds) {

  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  ganancias <- numeric(length(folds))
  niters    <- integer(length(folds))

  for (i in seq_along(folds)) {

    modelo_train <- lgb.train(
      data   = folds[[i]]$dtrain,
      valids = list(valid = folds[[i]]$dvalidate),
      eval   = "auc",
      param  = param_completo,
      verbose = -100
    )

    prob <- predict(modelo_train, folds[[i]]$matriz_validate)
    ganancias[i] <- calcular_ganancia_pico(prob, folds[[i]]$clase_real_validate)
    niters[i]    <- modelo_train$best_iter

    rm(modelo_train)
  }

  gc(full = TRUE, verbose = FALSE)
  list(mean(ganancias), as.integer(round(mean(niters))))
}

#### Comparacion 1: Manual (12 var) vs Original -- por PICO de ganancia

Ajuste sobre la version anterior: se compara el pico de cada curva de ganancia (el mejor punto de cada configuracion), no un corte fijo de envios -- asi ninguna configuracion queda en desventaja por tener su optimo mas lejos de un punto arbitrario comun. Mismo patron secuencial (MejorArbol) que 06/08, tope 15 semillas.

In [36]:
PARAM$comparacion$qsemillas_tope <- 15

if (!require("primes")) install.packages("primes")
require("primes")
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia)
semillas <- sample(primos, PARAM$comparacion$qsemillas_tope)

tb_comparacion_manual_gan <- data.table(semilla = integer(), GAN_original = numeric(), GAN_manual = numeric())
pvalue <- 1.0
isem <- 1

while ((isem <= PARAM$comparacion$qsemillas_tope) & (pvalue > 0.05)) {

  semilla <- semillas[isem]
  cat("\n=== semilla", isem, "de", PARAM$comparacion$qsemillas_tope, ":", semilla, "===\n")

  folds_original <- construir_folds_internos_ganancia(semilla, campos_buenos_original)
  res_original   <- Estimar_Ganancia_lightgbm(list(seed = semilla), folds_original)

  folds_manual   <- construir_folds_internos_ganancia(semilla, campos_buenos_manual)
  res_manual     <- Estimar_Ganancia_lightgbm(list(seed = semilla), folds_manual)

  tb_comparacion_manual_gan <- rbind(tb_comparacion_manual_gan, data.table(
    semilla = semilla, GAN_original = res_original[[1]], GAN_manual = res_manual[[1]]
  ))

  cat("  ganancia original:", res_original[[1]], " | ganancia manual:", res_manual[[1]], "\n")

  fwrite(tb_comparacion_manual_gan, file = "tb_comparacion_manual_ganancia_pico.txt", sep = "\t")

  if (nrow(tb_comparacion_manual_gan) >= 5) {
    wt <- wilcox.test(tb_comparacion_manual_gan$GAN_manual, tb_comparacion_manual_gan$GAN_original, paired = TRUE)
    pvalue <- wt$p.value
    cat("  n=", nrow(tb_comparacion_manual_gan), " p-value=", round(pvalue, 4), "\n")
  }

  rm(folds_original, folds_manual)
  gc(full = TRUE, verbose = FALSE)
  isem <- isem + 1
}

tb_comparacion_manual_gan

Loading required package: primes




=== semilla 1 de 15 : 422911 ===
  ganancia original: 82191667  | ganancia manual: 83308333 

=== semilla 2 de 15 : 590929 ===
  ganancia original: 82958333  | ganancia manual: 83158333 

=== semilla 3 de 15 : 516839 ===
  ganancia original: 84066667  | ganancia manual: 81841667 

=== semilla 4 de 15 : 206123 ===
  ganancia original: 83866667  | ganancia manual: 83425000 

=== semilla 5 de 15 : 831433 ===
  ganancia original: 83258333  | ganancia manual: 83475000 
  n= 5  p-value= 1 

=== semilla 6 de 15 : 990841 ===
  ganancia original: 83725000  | ganancia manual: 81058333 
  n= 6  p-value= 0.5625 

=== semilla 7 de 15 : 334349 ===
  ganancia original: 83608333  | ganancia manual: 83416667 
  n= 7  p-value= 0.5781 

=== semilla 8 de 15 : 793493 ===
  ganancia original: 83925000  | ganancia manual: 82641667 
  n= 8  p-value= 0.3125 

=== semilla 9 de 15 : 274529 ===
  ganancia original: 83308333  | ganancia manual: 82925000 
  n= 9  p-value= 0.2031 

=== semilla 10 de 15 : 215939 ===

semilla,GAN_original,GAN_manual
<int>,<dbl>,<dbl>
422911,82191667,83308333
590929,82958333,83158333
516839,84066667,81841667
206123,83866667,83425000
831433,83258333,83475000
990841,83725000,81058333
334349,83608333,83416667
793493,83925000,82641667
274529,83308333,82925000


In [37]:
wt_manual_gan <- wilcox.test(tb_comparacion_manual_gan$GAN_manual, tb_comparacion_manual_gan$GAN_original, paired = TRUE)
wt_manual_gan

cat("\nn semillas:", nrow(tb_comparacion_manual_gan), "\n")
cat("Ganancia promedio original:", round(mean(tb_comparacion_manual_gan$GAN_original)), "\n")
cat("Ganancia promedio manual:  ", round(mean(tb_comparacion_manual_gan$GAN_manual)), "\n")
cat("p-value:", round(wt_manual_gan$p.value, 5), "\n")


	Wilcoxon signed rank exact test

data:  tb_comparacion_manual_gan$GAN_manual and tb_comparacion_manual_gan$GAN_original
V = 46, p-value = 0.4543
alternative hypothesis: true location shift is not equal to 0



n semillas: 15 
Ganancia promedio original: 83206667 
Ganancia promedio manual:   83015000 
p-value: 0.45428 


#### Comparacion 2: Automatico (hojas RF) vs Original -- por PICO de ganancia

In [38]:
PARAM$rf_fe$num_trees <- 30
PARAM$rf_fe$max_depth <- 6

construir_fold_automatico_ganancia <- function(meses_train, mes_validate, campos_base, semilla) {

  fold_train_mask    <- dataset$foto_mes %in% meses_train &
    (dataset$clase_ternaria %in% c("BAJA+1", "BAJA+2") | dataset$azar < PARAM$trainingstrategy$training_pct)
  fold_validate_mask <- dataset$foto_mes == mes_validate

  set.seed(semilla)
  rf_fe <- ranger(
    x = dataset[fold_train_mask, campos_base, with = FALSE],
    y = as.factor(dataset[fold_train_mask, clase01]),
    num.trees = PARAM$rf_fe$num_trees, max.depth = PARAM$rf_fe$max_depth,
    num.threads = 0, seed = semilla
  )

  hojas_train    <- predict(rf_fe, data = dataset[fold_train_mask, campos_base, with = FALSE], type = "terminalNodes")$predictions
  hojas_validate <- predict(rf_fe, data = dataset[fold_validate_mask, campos_base, with = FALSE], type = "terminalNodes")$predictions
  colnames(hojas_train) <- colnames(hojas_validate) <- paste0("rf_hoja_arbol", seq_len(ncol(hojas_train)))

  matriz_validate <- cbind(data.matrix(dataset[fold_validate_mask, campos_base, with = FALSE]), hojas_validate)

  list(
    dtrain = lgb.Dataset(
      data  = cbind(data.matrix(dataset[fold_train_mask, campos_base, with = FALSE]), hojas_train),
      label = dataset[fold_train_mask, clase01],
      free_raw_data = TRUE
    ),
    dvalidate = lgb.Dataset(
      data  = matriz_validate,
      label = dataset[fold_validate_mask, clase01],
      free_raw_data = TRUE
    ),
    matriz_validate     = matriz_validate,
    clase_real_validate = dataset[fold_validate_mask, clase_ternaria]
  )
}

construir_folds_internos_automatico_ganancia <- function(semilla, campos_base) {
  set.seed(semilla, kind = "L'Ecuyer-CMRG")
  dataset[, azar := runif(nrow(dataset))]

  lapply(PARAM$trainingstrategy$validacion_interna, function(mes_val) {
    corte_train <- mes_menos(mes_val, PARAM$trainingstrategy$horizonte_meses)
    meses_train_fold <- PARAM$trainingstrategy$training[ PARAM$trainingstrategy$training <= corte_train ]
    construir_fold_automatico_ganancia(meses_train_fold, mes_val, campos_base, semilla)
  })
}

In [44]:
tb_comparacion_auto_gan <- fread("tb_comparacion_automatico_ganancia_pico.txt")
cat("Semillas ya procesadas:", nrow(tb_comparacion_auto_gan), "\n")
tb_comparacion_auto_gan

Semillas ya procesadas: 6 


semilla,GAN_original,GAN_automatico
<int>,<dbl>,<dbl>
107057,84000000,82775000
811771,82141667,81600000
754421,83533333,83483333
404837,84425000,82308333
137387,83150000,81950000
732491,83300000,82466667


In [45]:
# reconstruyo la MISMA secuencia de semillas que se uso originalmente (deterministica,
# depende de PARAM$semilla_primigenia -- si el kernel nunca murio, sigue siendo la misma)
set.seed(PARAM$semilla_primigenia)
semillas2 <- sample(primos, PARAM$comparacion$qsemillas_tope)

isem <- nrow(tb_comparacion_auto_gan) + 1

pvalue <- if (nrow(tb_comparacion_auto_gan) >= 5) {
  wilcox.test(tb_comparacion_auto_gan$GAN_automatico, tb_comparacion_auto_gan$GAN_original, paired = TRUE)$p.value
} else { 1.0 }

cat("Retomando desde la semilla", isem, "-- p-value actual:", round(pvalue, 4), "\n")

while ((isem <= PARAM$comparacion$qsemillas_tope) & (pvalue > 0.05)) {

  semilla <- semillas2[isem]
  cat("\n=== semilla", isem, "de", PARAM$comparacion$qsemillas_tope, ":", semilla, "===\n")

  folds_original   <- construir_folds_internos_ganancia(semilla, campos_buenos_original)
  res_original     <- Estimar_Ganancia_lightgbm(list(seed = semilla), folds_original)

  folds_automatico <- construir_folds_internos_automatico_ganancia(semilla, campos_buenos_original)
  res_automatico   <- Estimar_Ganancia_lightgbm(list(seed = semilla), folds_automatico)

  tb_comparacion_auto_gan <- rbind(tb_comparacion_auto_gan, data.table(
    semilla = semilla, GAN_original = res_original[[1]], GAN_automatico = res_automatico[[1]]
  ))

  cat("  ganancia original:", res_original[[1]], " | ganancia automatico:", res_automatico[[1]], "\n")

  fwrite(tb_comparacion_auto_gan, file = "tb_comparacion_automatico_ganancia_pico.txt", sep = "\t")

  if (nrow(tb_comparacion_auto_gan) >= 5) {
    wt <- wilcox.test(tb_comparacion_auto_gan$GAN_automatico, tb_comparacion_auto_gan$GAN_original, paired = TRUE)
    pvalue <- wt$p.value
    cat("  n=", nrow(tb_comparacion_auto_gan), " p-value=", round(pvalue, 4), "\n")
  }

  rm(folds_original, folds_automatico)
  gc(full = TRUE, verbose = FALSE)
  isem <- isem + 1
}

tb_comparacion_auto_gan

Retomando desde la semilla 7 -- p-value actual: 0.0313 


semilla,GAN_original,GAN_automatico
<int>,<dbl>,<dbl>
107057,84000000,82775000
811771,82141667,81600000
754421,83533333,83483333
404837,84425000,82308333
137387,83150000,81950000
732491,83300000,82466667


In [46]:
set.seed(PARAM$semilla_primigenia)
semillas2 <- sample(primos, PARAM$comparacion$qsemillas_tope)

tb_comparacion_auto_gan <- data.table(semilla = integer(), GAN_original = numeric(), GAN_automatico = numeric())
pvalue <- 1.0
isem <- 1

while ((isem <= PARAM$comparacion$qsemillas_tope) & (pvalue > 0.05)) {

  semilla <- semillas2[isem]
  cat("\n=== semilla", isem, "de", PARAM$comparacion$qsemillas_tope, ":", semilla, "===\n")

  folds_original   <- construir_folds_internos_ganancia(semilla, campos_buenos_original)
  res_original     <- Estimar_Ganancia_lightgbm(list(seed = semilla), folds_original)

  folds_automatico <- construir_folds_internos_automatico_ganancia(semilla, campos_buenos_original)
  res_automatico   <- Estimar_Ganancia_lightgbm(list(seed = semilla), folds_automatico)

  tb_comparacion_auto_gan <- rbind(tb_comparacion_auto_gan, data.table(
    semilla = semilla, GAN_original = res_original[[1]], GAN_automatico = res_automatico[[1]]
  ))

  cat("  ganancia original:", res_original[[1]], " | ganancia automatico:", res_automatico[[1]], "\n")

  fwrite(tb_comparacion_auto_gan, file = "tb_comparacion_automatico_ganancia_pico.txt", sep = "\t")

  if (nrow(tb_comparacion_auto_gan) >= 5) {
    wt <- wilcox.test(tb_comparacion_auto_gan$GAN_automatico, tb_comparacion_auto_gan$GAN_original, paired = TRUE)
    pvalue <- wt$p.value
    cat("  n=", nrow(tb_comparacion_auto_gan), " p-value=", round(pvalue, 4), "\n")
  }

  rm(folds_original, folds_automatico)
  gc(full = TRUE, verbose = FALSE)
  isem <- isem + 1
}

tb_comparacion_auto_gan


=== semilla 1 de 15 : 107057 ===
  ganancia original: 8.4e+07  | ganancia automatico: 82775000 

=== semilla 2 de 15 : 811771 ===
  ganancia original: 82141667  | ganancia automatico: 81600000 

=== semilla 3 de 15 : 754421 ===
  ganancia original: 83533333  | ganancia automatico: 83483333 

=== semilla 4 de 15 : 404837 ===
  ganancia original: 84425000  | ganancia automatico: 82308333 

=== semilla 5 de 15 : 137387 ===
  ganancia original: 83150000  | ganancia automatico: 81950000 
  n= 5  p-value= 0.0625 

=== semilla 6 de 15 : 732491 ===
  ganancia original: 83300000  | ganancia automatico: 82466667 
  n= 6  p-value= 0.0313 


semilla,GAN_original,GAN_automatico
<int>,<dbl>,<dbl>
107057,84000000,82775000
811771,82141667,81600000
754421,83533333,83483333
404837,84425000,82308333
137387,83150000,81950000
732491,83300000,82466667


In [47]:
wt_auto_gan <- wilcox.test(tb_comparacion_auto_gan$GAN_automatico, tb_comparacion_auto_gan$GAN_original, paired = TRUE)
wt_auto_gan

cat("\nn semillas:", nrow(tb_comparacion_auto_gan), "\n")
cat("Ganancia promedio original:  ", round(mean(tb_comparacion_auto_gan$GAN_original)), "\n")
cat("Ganancia promedio automatico:", round(mean(tb_comparacion_auto_gan$GAN_automatico)), "\n")
cat("p-value:", round(wt_auto_gan$p.value, 5), "\n")


	Wilcoxon signed rank exact test

data:  tb_comparacion_auto_gan$GAN_automatico and tb_comparacion_auto_gan$GAN_original
V = 0, p-value = 0.03125
alternative hypothesis: true location shift is not equal to 0



n semillas: 6 
Ganancia promedio original:   83425000 
Ganancia promedio automatico: 82430556 
p-value: 0.03125 


#### Conclusion comparada: AUC vs Ganancia como criterio de decision

In [48]:
cat("=== Manual (12 var) vs Original ===\n")
cat("  Por AUC (notebook 08):      p=0.103 (21 semillas combinadas) -> sin diferencia significativa\n")
cat("  Por Ganancia (pico de la curva): p=", round(wt_manual_gan$p.value, 4),
    " (", nrow(tb_comparacion_manual_gan), " semillas) -> ",
    ifelse(wt_manual_gan$p.value < 0.05, "SI hay diferencia significativa", "sin diferencia significativa"), "\n\n", sep="")

cat("=== Automatico (RF) vs Original ===\n")
cat("  Por AUC (notebook 07):      p=0.031 (6 semillas)  -> Original significativamente mejor\n")
cat("  Por Ganancia (pico de la curva): p=", round(wt_auto_gan$p.value, 4),
    " (", nrow(tb_comparacion_auto_gan), " semillas) -> ",
    ifelse(wt_auto_gan$p.value < 0.05, "SI hay diferencia significativa", "sin diferencia significativa"), "\n", sep="")

=== Manual (12 var) vs Original ===
  Por AUC (notebook 08):      p=0.103 (21 semillas combinadas) -> sin diferencia significativa
  Por Ganancia (pico de la curva): p=0.4543 (15 semillas) -> sin diferencia significativa

=== Automatico (RF) vs Original ===
  Por AUC (notebook 07):      p=0.031 (6 semillas)  -> Original significativamente mejor
  Por Ganancia (pico de la curva): p=0.0313 (6 semillas) -> SI hay diferencia significativa
